<a href="https://colab.research.google.com/github/Nourin-Nusrat/CSE4261_DNN/blob/main/Assignment1/DNNAssignment1_resnet101_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import warnings
import sys
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

In [ ]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers
import numpy as np

In [ ]:
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()

train_filter = (y_train_tmp < 20).flatten()
test_filter = (y_test_tmp < 20).flatten()

x_train = x_train_tmp[train_filter]
y_train = y_train_tmp[train_filter]
x_test = x_test_tmp[test_filter]
y_test = y_test_tmp[test_filter]

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

trainY = to_categorical(y_train, num_classes=20)
testY = to_categorical(y_test, num_classes=20)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [ ]:
input_shape = (32, 32, 3)
resnet101_model = keras.applications.ResNet101(
    include_top=False,
    weights="imagenet",
    input_shape=input_shape,
    pooling='avg'
)
model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        resnet101_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dropout(0.4),
        layers.Dense(127),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet101 (Functional)          │ (None, 2048)           │    42,658,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 127)            │        65,151 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 127)            │           508 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 127)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │         2,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 43,777,531 (167.00 MB)

 Trainable params: 43,670,909 (166.59 MB)

 Non-trainable params: 106,622 (416.49 KB)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

epochs = 10
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy',],
)

model.fit(x_train, trainY, epochs=epochs, callbacks=[early_stopping], validation_split=0.1)
model.save('resnet101_model.keras')

Epoch 1/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 181s 248ms/step - accuracy: 0.1869 - loss: 2.7566 - val_accuracy: 0.0430 - val_loss: 18.5107
Epoch 2/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 50s 65ms/step - accuracy: 0.4671 - loss: 1.7681 - val_accuracy: 0.0430 - val_loss: 109.6827
Epoch 3/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 66ms/step - accuracy: 0.5507 - loss: 1.5370 - val_accuracy: 0.2150 - val_loss: 5.3160
Epoch 4/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 17s 60ms/step - accuracy: 0.5466 - loss: 1.5203 - val_accuracy: 0.4660 - val_loss: 13.4066
Epoch 5/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 62ms/step - accuracy: 0.6232 - loss: 1.2620 - val_accuracy: 0.5560 - val_loss: 3.9409
Epoch 6/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 65ms/step - accuracy: 0.7032 - loss: 0.9783 - val_accuracy: 0.5680 - val_loss: 3.2467
Epoch 7/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 20s 62ms/step - accuracy: 0.7268 - loss: 0.9081 - val_accuracy: 0.4780 - val_loss: 57.3351
Epoch 8/10
282/282 ━━━━━━━━━━━━━━━━━━━━ 21s 63ms/step - accuracy: 0.6241 - loss: 1.

In [ ]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.1893 - loss: 2.6708


[2.6715729236602783, 0.19499999284744263]